# Type-B Training — CNN-3Layer & ResNet18 × TinyBERT Mean

Experiment 2: CNN architecture comparison using the best embedding from Experiment 1.

**Embedding fixed**: `tinybert_mean` (312-dim) — top-ranked in Stage 1  
**Models**: `cnn_3layer`, `resnet18_pt`  
**Loss**: `CombinedLoss` for cnn_3layer, `MSE` for resnet18_pt  

Supports two modes — set `MODE` in the Config cell:
- `'local'` : runs against local repo (VS Code + local kernel)
- `'colab'` : runs on Google Colab (mounts Drive, clones GitHub repo)

## Cell 1 — Config

In [ ]:
# ── MODE ──────────────────────────────────────────────────────────────────────
MODE = 'colab'   # 'local' or 'colab'

# ── Local config (used when MODE='local') ─────────────────────────────────────
LOCAL_REPO_DIR = '/Users/yeon/work/UoB_CourseWork/uob-ds-intro-to-ai-final-cw-2026'

# ── Colab config (used when MODE='colab') ─────────────────────────────────────
# Token is loaded from Colab Secrets (left sidebar → key icon → add GITHUB_TOKEN)
# Never hardcode the token here — GitHub will auto-revoke it.
GITHUB_REPO_URL = 'https://github.com/Rylie-KIM/uob-ds-intro-to-ai-final-cw-2026'
DRIVE_BASE      = '/content/drive/MyDrive/uob-ds-ai'

# ── Experiment config ─────────────────────────────────────────────────────────
# Embedding fixed to tinybert_mean — best performer in Stage 1 (Experiment 1).
# SKIP_EMBED=True: tinybert_mean_embedding_result_typeb.pt already exists.
EMBEDDING  = 'tinybert_mean'
SKIP_EMBED = True

# Per-model epoch budgets
# cnn_3layer   : trained from scratch → 30 epochs
# resnet18_pt  : ImageNet pretrained backbone → 20 epochs (converges faster)
MODEL_EPOCHS = {
    'cnn_3layer':  30,
    'resnet18_pt': 20,
}

# ── Run tag ──────────────────────────────────────────────────────────────────
import datetime as _dt
_BASE_TAG = 'exp2_cnn_arch'
RUN_TAG   = f'{_BASE_TAG}_{_dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")}'
# ─────────────────────────────────────────────────────────────────────────────

print(f'MODE    = {MODE}')
print(f'RUN_TAG = {RUN_TAG!r}')
print(f'EMBEDDING  = {EMBEDDING}')
print(f'SKIP_EMBED = {SKIP_EMBED}')
for m, e in MODEL_EPOCHS.items():
    print(f'  {m}: {e} epochs')

## Cell 2-1 — Check corrupted images (Colab only)

In [ ]:
from pathlib import Path
from PIL import Image

img_dir = Path('/content/images/type-b')
if img_dir.exists():
    corrupted = []
    for f in sorted(img_dir.glob('*.png')):
        try:
            Image.open(f).verify()
        except Exception:
            corrupted.append(f)
    print(f'Corrupted: {len(corrupted)} files')
    for f in corrupted:
        print(f'  deleting {f.name}')
        f.unlink()
    print('Done.')
else:
    print('Image dir not found — will be created in Cell 2-2.')

## Cell 2-2 — Setup (auto-selects local or Colab path)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path

if MODE == 'local':
    REPO_DIR = LOCAL_REPO_DIR
    print(f'Local mode. Repo: {REPO_DIR}')

elif MODE == 'colab':
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    REPO_DIR      = '/content/repo'
    DRIVE_DATA    = f'{DRIVE_BASE}/data'
    DRIVE_RESULTS = f'{DRIVE_BASE}/results'

    # Build authenticated clone URL from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url_auth = GITHUB_REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        print('WARNING: GITHUB_TOKEN not found in Colab Secrets.')
        repo_url_auth = GITHUB_REPO_URL

    # Clone or pull repo
    if not os.path.exists(REPO_DIR):
        print('Cloning repo...')
        subprocess.run(['git', 'clone', repo_url_auth, REPO_DIR], check=True)
    else:
        print('Repo exists. Pulling latest...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
        commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
        print(f'  HEAD: {commit}')

    # ── Copy train+val images to Colab local disk ────────────────────────────
    import pandas as _pd

    def _get_test_filenames(repo_dir: str) -> set:
        splits_csv    = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'splits' / 'type_b_splits_seed42.csv'
        image_map_csv = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'image_map_b.csv'
        sentences_csv = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'sentences_b.csv'
        splits_df = _pd.read_csv(splits_csv)
        image_map = _pd.read_csv(image_map_csv)
        sentences = _pd.read_csv(sentences_csv)
        df        = image_map.merge(sentences, on='sentence_id')
        test_idx  = splits_df[splits_df['split'] == 'test']['idx'].tolist()
        return set(df.iloc[test_idx]['filename'].tolist())

    test_filenames = _get_test_filenames(REPO_DIR)
    print(f'[info] {len(test_filenames)} test images excluded (evaluated locally)')

    LOCAL_IMG_BASE = Path('/content/images')
    for dtype in ['type-b']:
        local_img = LOCAL_IMG_BASE / dtype
        drive_img = Path(DRIVE_DATA) / 'images' / dtype
        repo_img  = Path(REPO_DIR) / 'src' / 'data' / 'images' / dtype

        local_img.mkdir(parents=True, exist_ok=True)
        all_drive_files = sorted(drive_img.glob('*.png'))
        total           = len(all_drive_files)
        src_files   = [f for f in all_drive_files if f.name not in test_filenames]
        local_files = set(f.name for f in local_img.glob('*.png'))
        missing     = [f for f in src_files if f.name not in local_files]

        print(f'[info] images/{dtype}: {total} total | {len(src_files)} train+val | {total - len(src_files)} test (skipped)')
        if not missing:
            print(f'[skip] images/{dtype} — all {len(src_files)} train+val files already on local disk')
        else:
            print(f'Copying {len(missing)}/{len(src_files)} images ({dtype}) Drive → /content/images/{dtype}/')
            t0 = time.time()
            for i, src in enumerate(missing, 1):
                shutil.copy2(str(src), str(local_img / src.name))
                if i % 500 == 0 or i == len(missing):
                    elapsed = time.time() - t0
                    eta     = elapsed / i * (len(missing) - i)
                    print(f'  {i}/{len(missing)} ({i/len(missing)*100:.0f}%)  elapsed={elapsed:.0f}s  eta={eta:.0f}s', end='\r')
            print(f'\n[done] {len(missing)} images copied in {time.time()-t0:.1f}s')

        if repo_img.is_symlink():
            os.unlink(str(repo_img))
        elif repo_img.exists():
            shutil.rmtree(str(repo_img))
        os.symlink(str(local_img), str(repo_img))
        print(f'[symlinked] repo/src/data/images/{dtype} → /content/images/{dtype}')

    # ── Restore results from Drive → local repo ───────────────────────────────
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        drive_sub = Path(DRIVE_RESULTS) / sub
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        repo_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in drive_sub.glob('*'):
            if f.is_file() and not (repo_sub / f.name).exists():
                shutil.copy2(str(f), str(repo_sub / f.name))
                copied += 1
        print(f'[restored] results/{sub}  ({copied} files from Drive)')

    # ── Copy tinybert_mean .pt from Drive ─────────────────────────────────────
    drive_emb = Path(DRIVE_RESULTS) / 'embeddings'
    drive_emb.mkdir(parents=True, exist_ok=True)
    repo_emb = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results'
    repo_emb.mkdir(parents=True, exist_ok=True)
    pt_name = f'{EMBEDDING}_embedding_result_typeb.pt'
    pt_drive = drive_emb / pt_name
    pt_repo  = repo_emb / pt_name
    if not pt_repo.exists():
        if pt_drive.exists():
            shutil.copy2(str(pt_drive), str(pt_repo))
            print(f'[copied] {pt_name} Drive → repo')
        else:
            print(f'WARNING: {pt_name} not found in Drive — check DRIVE_RESULTS/embeddings/')
    else:
        print(f'[exists] {pt_name}')

    print(f'\nColab mode ready. Repo: {REPO_DIR}')

# Add repo to sys.path
for p in [REPO_DIR, str(Path(REPO_DIR) / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 3 — Install Dependencies

In [ ]:
if MODE == 'colab':
    !pip install -q sentence-transformers gensim python-docx datasets
    print('Dependencies installed.')
else:
    print('Local mode — using existing venv. Skipping pip install.')

## Cell 4 — Verify Paths

In [ ]:
pt_path = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results' / f'{EMBEDDING}_embedding_result_typeb.pt'

checks = {
    'sentences_b.csv'  : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'sentences_b.csv',
    'image_map_b.csv'  : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'image_map_b.csv',
    'images/type-b/'   : Path(REPO_DIR) / 'src' / 'data' / 'images' / 'type-b',
    'splits CSV'       : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'splits',
    'train_type_b.py'  : Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
    'cnn_3layer.py'    : Path(REPO_DIR) / 'src' / 'models' / 'type-b' / 'cnn_3layer.py',
    'resnet18_pt.py'   : Path(REPO_DIR) / 'src' / 'models' / 'type-b' / 'resnet18_pt.py',
    f'{EMBEDDING}.pt'  : pt_path,
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f'  {"OK" if ok else "MISSING"}  {name:30s}  {path}')
    if not ok:
        all_ok = False
print()
print('All paths OK — ready to train.' if all_ok else 'Fix missing paths before continuing.')

## Cell 5 — Train

Runs `cnn_3layer` and `resnet18_pt` sequentially with `tinybert_mean` embeddings.  
Results are backed up to Drive after each model (Colab mode only).

| Model | Loss | Epochs | Notes |
|---|---|---|---|
| `cnn_3layer` | CombinedLoss (MSE + Cosine, α=0.5) | 30 | Trained from scratch |
| `resnet18_pt` | MSELoss | 20 | ImageNet pretrained backbone; converges faster |

In [ ]:
import importlib.util as _ilu

# Evict any stale cached config modules
for _key in list(sys.modules.keys()):
    if 'config.paths' in _key or _key in ('src.config', 'config'):
        del sys.modules[_key]

_spec = _ilu.spec_from_file_location(
    'train_type_b',
    Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

print(f'[paths] METRICS_B     = {_mod.METRICS_B}')
print(f'[paths] CHECKPOINTS_B = {_mod.CHECKPOINTS_B}')
assert str(_mod.METRICS_B).endswith('metrics/type-b'), \
    f'METRICS_B path wrong: {_mod.METRICS_B}'
assert str(_mod.CHECKPOINTS_B).endswith('checkpoints/type-b'), \
    f'CHECKPOINTS_B path wrong: {_mod.CHECKPOINTS_B}'
print('[paths] OK\n')


def _backup_results_to_drive():
    if MODE != 'colab':
        return
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in sorted(repo_sub.glob('*')):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                copied += 1
        print(f'  [backup] results/{sub} ({copied} files)')


total = len(MODEL_EPOCHS)
for n, (model_name, epochs) in enumerate(MODEL_EPOCHS.items(), 1):
    print(f'\n[{n}/{total}] {model_name} × {EMBEDDING}  epochs={epochs}  tag={RUN_TAG!r}')
    _mod.run_experiment(
        model_name=model_name,
        embedding_name=EMBEDDING,
        epochs=epochs,
        device=device,
        run_tag=RUN_TAG,
    )
    _backup_results_to_drive()

print('\nAll done.')

## Cell 5.5 — (Optional) Manual re-backup to Drive

In [ ]:
if MODE == 'colab':
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in repo_sub.glob('*'):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                copied += 1
        print(f'[backed up] results/{sub} → Drive  ({copied} files)')
else:
    print('Local mode — Drive backup skipped.')

## Cell 6 — Training Curves

In [ ]:
import pandas as pd

metrics_dir = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / 'metrics' / 'type-b'

for model_name in MODEL_EPOCHS:
    run_name = f'b_{model_name}_{EMBEDDING}_{RUN_TAG}'
    print(f'\n{"="*60}')
    print(f'  {run_name}')
    print(f'{"="*60}')

    log_path = metrics_dir / f'{run_name}_training_log.csv'
    if log_path.exists():
        log = pd.read_csv(log_path)
        print(f'[training_log]  {log_path.name}  ({len(log)} epochs)')
        display(log)
    else:
        print(f'[missing] {log_path.name}')

## Cell 7 — Push Results to GitHub

In [ ]:
COMMIT_MSG  = f'Add results: cnn_3layer + resnet18_pt x {EMBEDDING}  tag={RUN_TAG}'
_push_script = str(Path(REPO_DIR) / 'notebooks' / 'train-evaluation' / 'type-b' / '_push_results.py')
%run -i $_push_script